# 01 — Inspeção e EDA

Estudo de caso simulado com PaySim. Execute a partir da raiz do projeto ou dentro de `notebooks/`. Requer o CSV original em `data/raw/`; este pacote não inclui a base.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data.prepare_data import find_csv
csv_path = find_csv(ROOT / 'data' / 'raw')
df = pd.read_csv(csv_path)
print(csv_path, df.shape)


## Perfil, tipos e qualidade
Sem dados imputados ou limpeza silenciosa; primeiro inspecione e registre.

In [ ]:
display(df.head())
profile = pd.DataFrame({'tipo':df.dtypes.astype(str),'preenchidos':df.notna().sum(),'ausentes':df.isna().sum(),'unicos':df.nunique(dropna=True)})
display(profile)
print('Duplicatas:', int(df.duplicated().sum()))
print('Valores ausentes:', int(df.isna().sum().sum()))
display(df.describe(include='all').T)


## Distribuição do alvo
A fraude é classe minoritária; acurácia isolada seria inadequada. As taxas descrevem apenas esta base sintética.

In [ ]:
counts=df['isFraud'].value_counts().sort_index()
display(pd.DataFrame({'quantidade':counts,'percentual':counts/len(df)*100}))
ax=counts.plot(kind='bar',color=['#577590','#f94144'],title='Transações por rótulo (PaySim)')
ax.set_xlabel('isFraud'); ax.set_ylabel('Transações'); plt.show()


## Limitações e vazamento
`isFraud` é o rótulo. Exclua `isFlaggedFraud` do baseline principal, pois é sinalização pré-existente; evite saldos posteriores (`newbalanceOrig`, `newbalanceDest`). `step` é tempo sintético, não calendário. Não extrapole para operações reais.
